# US Visa Dataset — Data Drift Monitoring with Evidently

This notebook generates a **Data Drift Report/Dashboard** for the U.S. visa dataset using **Evidently**.

✅ Works in VS Code / Jupyter by **saving an HTML report** and embedding it inline (instead of relying on `dashboard.show()`).

In [3]:
# (Optional) Install / upgrade Evidently
# If you already have Evidently installed in your environment, you can skip this cell.
# !pip install -U evidently packaging

In [2]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="scipy")  # hides harmless SciPy divide-by-zero warnings

import pandas as pd
from pathlib import Path

In [4]:
# ---- Load dataset ----
# Put Visadataset.csv in the same folder as this notebook (or update the path below).
DATA_PATH = Path("Visadataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Can't find {DATA_PATH}. Put Visadataset.csv next to this notebook, "
        "or update DATA_PATH to the correct file path."
    )

df = pd.read_csv(DATA_PATH)
df.head()

,case_id,continent,education_of_employee,has_job_experience,requires_job_training,no_of_employees,yr_of_estab,region_of_employment,prevailing_wage,unit_of_wage,full_time_position,case_status
0,EZYV01,Asia,High School,N,N,14513,2007,West,592.2029,Hour,Y,Denied
1,EZYV02,Asia,Master's,Y,N,2412,2002,Northeast,83425.6500,Year,Y,Certified
2,EZYV03,Asia,Bachelor's,N,Y,44444,2008,West,122996.8600,Year,Y,Denied
3,EZYV04,Asia,Bachelor's,N,N,98,1897,West,83434.0300,Year,Y,Denied
4,EZYV05,Africa,Master's,Y,N,1082,2005,South,149907.3900,Year,Y,Certified


## Basic cleanup / dtype fixes

We drop identifiers and ensure numeric columns are numeric.  
This reduces drift-test noise and prevents some statistical edge-cases.

In [5]:
# drop ID-like column if present (it adds noise and can break drift tests)
for col in ["case_id", "id"]:
    if col in df.columns:
        df = df.drop(columns=[col])

# coerce common numeric columns (safe even if they are already numeric)
NUM_COLS = ["no_of_employees", "yr_of_estab", "prevailing_wage"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# optionally drop rows with missing numeric essentials
# (you can comment these out if you prefer imputation)
df = df.dropna(subset=[c for c in NUM_COLS if c in df.columns]).reset_index(drop=True)

df.dtypes

continent                 object
education_of_employee     object
has_job_experience        object
requires_job_training     object
no_of_employees            int64
yr_of_estab                int64
region_of_employment      object
prevailing_wage          float64
unit_of_wage              object
full_time_position        object
case_status               object
dtype: object

## Create reference vs current datasets

Your earlier split `df[:200]` vs `df[200:]` can cause **missing categories** in the reference sample,
and Chi-square based tests may show SciPy warnings.

So we:
- sample a larger reference set
- sample a disjoint current set

In [6]:
# Choose sample sizes (adjust if you want)
REF_N = min(5000, len(df) // 2)
CUR_N = min(5000, len(df) - REF_N)

reference = df.sample(n=REF_N, random_state=42)
current = df.drop(reference.index).sample(n=CUR_N, random_state=43)

reference.shape, current.shape

((5000, 11), (5000, 11))

## Generate drift report

Evidently has multiple APIs across versions.  
This cell auto-detects your Evidently version and produces an HTML report:
- **New API (Report + DataDriftPreset)** for newer versions
- **Old API (Dashboard + DataDriftTab)** for older versions (like `0.2.8`)

In [7]:
from packaging import version

import evidently

out_html = "usvisa_data_drift_report.html"

ev = version.parse(getattr(evidently, "__version__", "0.0.0"))
print("Evidently version:", ev)

if ev >= version.parse("0.4.0"):
    # Newer Evidently API
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset

    report = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=reference, current_data=current)
    report.save_html(out_html)

else:
    # Older Evidently API (e.g., 0.2.8)
    from evidently.dashboard import Dashboard
    from evidently.tabs import DataDriftTab

    dashboard = Dashboard(tabs=[DataDriftTab()])
    dashboard.calculate(reference, current)
    dashboard.save(out_html)

print("Saved:", out_html)

Evidently version: 0.2.8


/opt/anaconda3/envs/visa/lib/python3.8/site-packages/evidently/analyzers/__init__.py:3: UserWarning: analyzers are deprecated, use metrics instead
  warnings.warn("analyzers are deprecated, use metrics instead")
/opt/anaconda3/envs/visa/lib/python3.8/site-packages/evidently/dashboard/__init__.py:8: UserWarning: dashboards are deprecated, use metrics instead
  warnings.warn("dashboards are deprecated, use metrics instead")
/opt/anaconda3/envs/visa/lib/python3.8/site-packages/evidently/tabs/__init__.py:8: UserWarning: 'import evidently.tabs' is deprecated, use 'import evidently.dashboard.tabs'
  warnings.warn(


Saved: usvisa_data_drift_report.html


## View the report inline

If you are in VS Code / Jupyter, this should render the HTML inside the notebook.

In [8]:
from IPython.display import IFrame
IFrame(out_html, width="100%", height=650)

### Optional: simulate drift (for a stronger demo)

If your real `current` sample looks similar to `reference`, drift may be low.
You can artificially simulate drift like this:

- increase wages
- shift employee counts
- change distribution of a categorical column

(Use only for demonstration.)

In [ ]:
# Uncomment to simulate drift and re-run the report cell above.

# current_drift = current.copy()
# if "prevailing_wage" in current_drift.columns:
#     current_drift["prevailing_wage"] = current_drift["prevailing_wage"] * 1.25
# if "no_of_employees" in current_drift.columns:
#     current_drift["no_of_employees"] = current_drift["no_of_employees"] * 1.10
# current = current_drift